# SU(3) exact O(u⁶) geometry and local-channel census

Run all cells without edits. The first code cell locates or requests the complete launch bundle. The long scan is CPU-only and checkpointed.

In [1]:
from pathlib import Path
import zipfile
candidates=list(Path('/content').glob('SU3_Y6_STAGE0_LAUNCH_BUNDLE*.zip'))
if not candidates:
    from google.colab import files
    uploaded=files.upload()
    candidates=[Path('/content')/name for name in uploaded if name.endswith('.zip')]
if not candidates:
    raise FileNotFoundError('Upload SU3_Y6_STAGE0_LAUNCH_BUNDLE_2026-06-14.zip')
bundle=candidates[0]
root=Path('/content/SU3_Y6_STAGE0_PACKAGE')
if root.exists():
    import shutil; shutil.rmtree(root)
root.mkdir(parents=True)
with zipfile.ZipFile(bundle) as zf: zf.extractall(root)
# Handle either flat or one-directory archive layouts.
files=list(root.rglob('su3_y6_stage0_launcher.py'))
if len(files)!=1: raise RuntimeError(files)
pkg=files[0].parent
print('package:',pkg)
print('bundle:',bundle)


Saving SU3_Y6_STAGE0_LAUNCH_BUNDLE_2026-06-14.zip to SU3_Y6_STAGE0_LAUNCH_BUNDLE_2026-06-14.zip
package: /content/SU3_Y6_STAGE0_PACKAGE/SU3_Y6_STAGE0_PACKAGE
bundle: /content/SU3_Y6_STAGE0_LAUNCH_BUNDLE_2026-06-14.zip


In [8]:
!unzip -q /content/SU3_Y6_STAGE0_RESUMABLE_V2_RELEASE.zip -d /content

%cd /content/SU3_Y6_STAGE0_RESUMABLE_V2

!python diagnose_y6_run.py \
    --root /content/SU3_Y6_RUN

/content/SU3_Y6_STAGE0_RESUMABLE_V2
root: /content/SU3_Y6_RUN exists: True
disk: usage(total=242486079488, used=22068604928, free=220400697344)

Processes:
  37236       00:01  6.0  0.1 13508 python3 diagnose_y6_run.py --root /content/SU3_Y6_RUN

Binary artifacts:
/content/SU3_Y6_RUN/MERGED/su3_y6_ordered.bin {'records': 4896498941781335, 'order': 2450840, 'size': 73166400, 'expected': 48002141865901468285612, 'valid': False}
/content/SU3_Y6_RUN/MERGED/su3_y6_unordered_merged.bin {'records': 4896498941781335, 'order': 2450840, 'size': 604440, 'expected': 48002141865901468285612, 'valid': False}
/content/SU3_Y6_RUN/SELF_TEST/chunk_0.bin None
/content/SU3_Y6_RUN/SIGNATURES/su3_y6_final_ordered_signatures.bin {'records': 144115188075855871, 'order': 4294967295, 'size': 22960, 'expected': 2475880077994299780314955792, 'valid': False}
/content/SU3_Y6_RUN/chunks/chunk_0.bin None
/content/SU3_Y6_RUN/chunks/chunk_1.bin {'records': 4896498941781335, 'order': 2450840, 'size': 480, 'expected': 48

In [9]:
!python su3_y6_stage0_launcher_v2.py \
    --root /content/SU3_Y6_RUN \
    --threads 2 \
    --chunk 100000 \
    --scan-chunk 250000

[2026-06-15T02:09:24] disk free: 205.26 GiB of 225.83 GiB
[2026-06-15T02:09:24] RUN: g++ -O3 -std=c++17 /content/SU3_Y6_STAGE0_RESUMABLE_V2/y6_seed_supports.cpp -o /content/SU3_Y6_RUN/bin/y6_seed_supports
[2026-06-15T02:09:27] RUN: g++ -O3 -std=c++17 /content/SU3_Y6_STAGE0_RESUMABLE_V2/y6_expand_support_shard.cpp -o /content/SU3_Y6_RUN/bin/y6_expand_support_shard
[2026-06-15T02:09:29] RUN: g++ -O3 -std=c++17 /content/SU3_Y6_STAGE0_RESUMABLE_V2/y6_merge_support_shards.cpp -o /content/SU3_Y6_RUN/bin/y6_merge_support_shards
[2026-06-15T02:09:30] RUN: g++ -O3 -std=c++17 /content/SU3_Y6_STAGE0_RESUMABLE_V2/y6_scan_support_shard.cpp -o /content/SU3_Y6_RUN/bin/y6_scan_support_shard
[2026-06-15T02:09:32] RUN: g++ -O3 -std=c++17 /content/SU3_Y6_STAGE0_RESUMABLE_V2/y6_ordered_words.cpp -o /content/SU3_Y6_RUN/bin/y6_ordered_words
[2026-06-15T02:09:34] RUN: /content/SU3_Y6_RUN/bin/y6_seed_supports --target 5 --self-test --output /content/SU3_Y6_RUN/y5_seed.bin --summary /content/SU3_Y6_RUN/y5_seed

In [11]:
%%bash
set -euo pipefail

cd /content

apt-get install -y zstd

sha256sum \
  SU3_Y6_RUN/y5_seed.bin \
  SU3_Y6_RUN/y6_supports.bin \
  SU3_Y6_RUN/final/y6_triality_survivors.tsv \
  SU3_Y6_RUN/final/y6_ordered_transition_words.tsv \
  > SU3_Y6_RUN/STAGE0_SHA256SUMS.txt

tar -I 'zstd -10 -T0' \
  -cf SU3_Y6_STAGE0_COMPLETE_2026-06-15.tar.zst \
  SU3_Y6_RUN/y5_seed_summary.json \
  SU3_Y6_RUN/y6_supports.bin \
  SU3_Y6_RUN/final \
  SU3_Y6_RUN/STAGE0_SHA256SUMS.txt \
  SU3_Y6_RUN/logs

ls -lh SU3_Y6_STAGE0_COMPLETE_2026-06-15.tar.zst
sha256sum SU3_Y6_STAGE0_COMPLETE_2026-06-15.tar.zst

/bin/sh: 1: zstd: not found
tar: SU3_Y6_STAGE0_COMPLETE_2026-06-15.tar.zst: Wrote only 4096 of 10240 bytes
tar: Child returned status 127
tar: Error is not recoverable: exiting now


CalledProcessError: Command 'b"set -euo pipefail\n\ncd /content\n\nsha256sum \\\n  SU3_Y6_RUN/y5_seed.bin \\\n  SU3_Y6_RUN/y6_supports.bin \\\n  SU3_Y6_RUN/final/y6_triality_survivors.tsv \\\n  SU3_Y6_RUN/final/y6_ordered_transition_words.tsv \\\n  > SU3_Y6_RUN/STAGE0_SHA256SUMS.txt\n\ntar -I 'zstd -10 -T0' \\\n  -cf SU3_Y6_STAGE0_COMPLETE_2026-06-15.tar.zst \\\n  SU3_Y6_RUN/y5_seed_summary.json \\\n  SU3_Y6_RUN/y6_supports.bin \\\n  SU3_Y6_RUN/final \\\n  SU3_Y6_RUN/STAGE0_SHA256SUMS.txt \\\n  SU3_Y6_RUN/logs\n\nls -lh SU3_Y6_STAGE0_COMPLETE_2026-06-15.tar.zst\nsha256sum SU3_Y6_STAGE0_COMPLETE_2026-06-15.tar.zst\n"' returned non-zero exit status 2.

In [12]:
%%bash
cd /content

zip -9 -r SU3_Y6_STAGE0_ESSENTIAL_RESULTS.zip \
  SU3_Y6_RUN/final \
  SU3_Y6_RUN/STAGE0_SHA256SUMS.txt \
  SU3_Y6_RUN/y5_seed_summary.json \
  SU3_Y6_RUN/run_state.json \
  SU3_Y6_RUN/ordered_merge.log \
  SU3_Y6_RUN/signature_census.log

  adding: SU3_Y6_RUN/final/ (stored 0%)
  adding: SU3_Y6_RUN/final/y6_triality_survivors.tsv (deflated 96%)
  adding: SU3_Y6_RUN/final/y6_ordered_transition_words.tsv (deflated 95%)
  adding: SU3_Y6_RUN/STAGE0_SHA256SUMS.txt (deflated 36%)
  adding: SU3_Y6_RUN/y5_seed_summary.json (deflated 21%)
  adding: SU3_Y6_RUN/run_state.json (deflated 43%)
  adding: SU3_Y6_RUN/ordered_merge.log (deflated 80%)
  adding: SU3_Y6_RUN/signature_census.log (deflated 68%)


In [3]:
import os, subprocess, sys, time
run_root='/content/SU3_Y6_RUN'
threads=min(32,os.cpu_count() or 8)
cmd=[sys.executable,str(pkg/'su3_y6_stage0_launcher.py'),'--root',run_root,'--threads',str(threads),'--chunk','100000']
print('threads:',threads)
print('command:',' '.join(cmd))
t0=time.time()
try:
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print(f"Error executing command: {e}")
    print(f"Stdout: {e.stdout}")
    print(f"Stderr: {e.stderr}")
    raise # Re-raise the exception after printing details
print('elapsed seconds:',time.time()-t0)

threads: 2
command: /usr/bin/python3 /content/SU3_Y6_STAGE0_PACKAGE/SU3_Y6_STAGE0_PACKAGE/su3_y6_stage0_launcher.py --root /content/SU3_Y6_RUN --threads 2 --chunk 100000
Error executing command: Command '['/usr/bin/python3', '/content/SU3_Y6_STAGE0_PACKAGE/SU3_Y6_STAGE0_PACKAGE/su3_y6_stage0_launcher.py', '--root', '/content/SU3_Y6_RUN', '--threads', '2', '--chunk', '100000']' returned non-zero exit status 1.
Stdout: + g++ -O3 -std=c++17 -pthread /content/SU3_Y6_STAGE0_PACKAGE/SU3_Y6_STAGE0_PACKAGE/su3_y6_stage0_census.cpp -o /content/SU3_Y6_RUN/su3_y6_stage0_census
+ g++ -O3 -std=c++17 /content/SU3_Y6_STAGE0_PACKAGE/SU3_Y6_STAGE0_PACKAGE/su3_y6_ordered_merge.cpp -o /content/SU3_Y6_RUN/su3_y6_ordered_merge
+ g++ -O3 -std=c++17 /content/SU3_Y6_STAGE0_PACKAGE/SU3_Y6_STAGE0_PACKAGE/su3_y6_signature_census.cpp -o /content/SU3_Y6_RUN/su3_y6_signature_census
+ /content/SU3_Y6_RUN/su3_y6_stage0_census --input /content/SU3_Y6_RUN/y5_connected_supports.bin --output-dir /content/SU3_Y6_RUN/SELF_

CalledProcessError: Command '['/usr/bin/python3', '/content/SU3_Y6_STAGE0_PACKAGE/SU3_Y6_STAGE0_PACKAGE/su3_y6_stage0_launcher.py', '--root', '/content/SU3_Y6_RUN', '--threads', '2', '--chunk', '100000']' returned non-zero exit status 1.

In [ ]:
from pathlib import Path

file_path = Path('/content/SU3_Y6_RUN/SIGNATURES/su3_y6_final_ordered_signatures.bin')

if file_path.exists():
    print(f"File exists: {file_path}")
    print(f"File size: {file_path.stat().st_size} bytes")
    with open(file_path, 'rb') as f:
        first_bytes = f.read(100) # Read the first 100 bytes
        print("First 100 bytes (hex):", first_bytes.hex())
else:
    print(f"File does not exist: {file_path}")

In [ ]:
from pathlib import Path
import json
summary=Path('/content/SU3_Y6_RUN/SU3_Y6_STAGE0/su3_y6_stage0_summary.json')
local=Path('/content/SU3_Y6_RUN/SU3_Y6_STAGE1_LOCAL/su3_y6_local_channel_summary.json')
print(summary.read_text())
print(local.read_text())
result=Path('/content/SU3_Y6_STAGE0_AND_LOCAL_RESULTS.zip')
print('RESULT BUNDLE:',result)


In [ ]:
# Optional download after the full run.
from google.colab import files
files.download('/content/SU3_Y6_STAGE0_AND_LOCAL_RESULTS.zip')
